In [ ]:
import sys, os, json
from pathlib import Path 
import torch
from transformers import AutoTokenizer,AutoModelForTokenClassification
PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from medical_transcriptions_templates import transcriptions
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"


# Load label mappings from the local JSON (same labels across all versions)
with open("../v01/data/id2label.json", "r") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

label2id = {v: k for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} labels: {id2label}")


In [ ]:
device

In [ ]:
def inference_wrapper(
    text: str,
    model,
    tokenizer,
    id2label,
    device,
    verbose=1

    ):
    tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
            text=text,
            model=model,
            tokenizer=tokenizer,
            id2label=id2label,
            device=device,
        )
    spans = word_labels_to_spans(text=text, word_offsets=word_offsets, word_labels=word_labels)
    if verbose:
        print("SPANS:")
        for s in spans:
            print(f"\t{s}")

    return spans

# V03

In [ ]:
VERSION = "v03"
RUN_IDX = "2"
LOCAL_MODEL_DIR = f"{PROJECT_ROOT}/{VERSION}/downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_{RUN_IDX}"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model_03 = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
model_03.to(device)
model_03.eval()
print(f"✅ Loaded model and tokenizer from {LOCAL_MODEL_DIR}")

In [ ]:

level = transcriptions[0]['level']
template = transcriptions[0]['template']
entities_neg = transcriptions[0]['entities']['SYMPTOM_NEG']
entities_pos = transcriptions[0]['entities']['SYMPTOM_POS']
print(f"Level inspected: {level}\nTEMPLATE:\n")
print(template)
print(f"Ground truth:\n\tSYMPTOM_NEG: {entities_neg}\n\tSYMPTOM_POS: {entities_pos}")

print("\nPREDICTIONS:")
spans = inference_wrapper(
    text=template,
    model=model_03,
    tokenizer=tokenizer,
    id2label=id2label,
    device=device,
    verbose=0
)
print("Predicted symptoms:")
for s in spans: 
    print(s) if s['label'] != 'O' else ''

print(f"\nAll spans {len(spans)}:")
spans

# v032

In [ ]:
VERSION = "v032"
RUN_IDX = "2"
LOCAL_MODEL_DIR = f"./downloaded_models/dmis-lab/biobert-base-cased-v1.1/{VERSION}/run_{RUN_IDX}"
model_v032 = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model_v032 = model_v032.to(device)
model_v032.eval()
print(f"Using device: {device}")

In [ ]:
level = transcriptions[0]['level']
template = transcriptions[0]['template']
entities_neg = transcriptions[0]['entities']['SYMPTOM_NEG']
entities_pos = transcriptions[0]['entities']['SYMPTOM_POS']
print(f"Level inspected: {level}\nTEMPLATE:\n")
print(template)
print(f"Ground truth:\n\tSYMPTOM_NEG: {entities_neg}\n\tSYMPTOM_POS: {entities_pos}")

print("\nPREDICTIONS:")
spans = inference_wrapper(
    text=template,
    model=model_v032,
    tokenizer=tokenizer,
    id2label=id2label,
    device=device,
    verbose=0
)
print("Predicted symptoms:")
for s in spans: 
    print(s) if s['label'] != 'O' else ''

print(f"\nAll spans {len(spans)}:")
spans